In [7]:
# First, install the CIDEr evaluation package (if not already installed)
# pip install git+https://github.com/michelecafagna26/cider.git#egg=cidereval

from cidereval import cider  # or ciderD if you prefer the defended version

def table_to_text(table):
    """
    Convert a table (list of lists or pandas DataFrame) to a string.
    For simplicity, we convert the table into a CSV-like format.
    """
    # If table is a list of lists:
    lines = []
    for row in table:
        # Convert each cell to string and join by commas
        line = ', '.join(str(cell) for cell in row)
        lines.append(line)
    # Join rows with newline
    return '\n'.join(lines)

# Example candidate and reference tables
candidate_table = [
    ["Name", "Age", "City"],
    ["Alice", 30, "New York"],
    ["Bob", 25, "Los Angeles"]
]

reference_table = [
    ["Name", "Age", "City"],
    ["Alice", 30, "New York"],
    ["Bob", 25, "Los Angeles"]
]

# Convert tables to text
cand_text = table_to_text(candidate_table)
ref_text = table_to_text(reference_table)

# Prepare the inputs for CIDEr:
# The CIDEr function expects the inputs as dictionaries mapping an ID to a list of sentences.
# Here, we use a single ID (say "table1") and put the entire table string in a list.
predictions = {"table1": [cand_text]}
references = {"table1": [ref_text]}  # if you have multiple reference tables, add them to the list

# Compute the CIDEr score
scores = cider(predictions=predictions, references=references, df="coco-val-df")

print("CIDEr score:", scores["avg_score"])


PTBTokenizer tokenized 11 tokens at 413.67 tokens per second.
PTBTokenizer tokenized 1 tokens at 39.86 tokens per second.


Error retrieveing coco-val-df.p df_mode set to 'coco-val'
CIDEr score: 0.0


In [8]:

def table_to_dict_list_comparison(table_string, suffix=""):
    table_string = table_string.replace("markdown", "")
    # Split the table into lines

    lines = table_string.strip().split('\n')

    # Find the first line that contains the table header (i.e., a line with '|')
    try:
        table_start_idx = next((i for i, line in enumerate(
            lines) if '|' in line), None)
    except:
        print("Error in table")
        print(table_string)

    # If no table is found, return an empty list
    if table_start_idx is None:
        return []

    # Process the header from the detected table start line
    header = lines[table_start_idx].strip().split('|')
    header = [col.strip() for col in header if col.strip()]

    # Prepare the list to hold dictionaries
    table_as_dicts = []

    # Loop through each data row, skipping any separator rows and stopping at ``` or blank lines
    for line in lines[table_start_idx + 1:]:
        # Stop processing if the table ends
        if '```' in line or not line.strip():
            break

        # Skip lines that contain only '---'
        if '---' in line:
            continue

        row_values = line.strip().split('|')
        row_values = [val.strip() for val in row_values if val.strip()]

        # Create a dictionary for the current row, ensuring to match header order with values
        row_dict = {}
        for i in range(len(header)):
            if i < len(row_values):
                row_dict[header[i]] = row_values[i]
            else:
                row_dict[header[i]] = None  # Fill with None if data is missing

        table_as_dicts.append(row_dict)

    return table_as_dicts


In [11]:
def compute_scores_for_tables_cider(tables):
    """
    Compute CIDEr-based scores for each table against its perturbations.

    Each table is assumed to be a dictionary with a key "original"
    and keys "pertubation0", "pertubation1", ..., "pertubation4".
    The helper function `table_to_dict_list_comparison` converts a table
    into the required format. The function `compute_cider_for_tables`
    computes the CIDEr score between two such formatted tables.

    Args:
        tables (list): List of dictionaries representing tables.

    Returns:
        list: A list of dictionaries with the original table data and
              additional keys 'result0' to 'result4' with the corresponding CIDEr scores.
    """
    score_results = []
    for table in tables:
        scores = {}
        original_table = table["original"]
        # Convert the original table once.
        orig_dict = table_to_dict_list_comparison(original_table)
        for i in range(5):
            perturbation = table[f'pertubation{i}']
            pert_dict = table_to_dict_list_comparison(perturbation)
            # Compute CIDEr score between the original and the perturbation.
            score = cider(table_to_text(orig_dict), pert_dict)
            scores[f'result{i}'] = score
        # Merge the original table info with the computed scores.
        score_results.append({**table, **scores})
    return score_results


In [12]:
import json 
with open("/home/turning/Jainit/TANQ/eval_dataset/tables.json") as f:
    tables = json.load(f)

In [13]:
compute_scores_for_tables_cider(tables)

PTBTokenizer tokenized 54 tokens at 1647.92 tokens per second.


AttributeError: 'dict' object has no attribute 'replace'